# Phase 3: Alpha Research & Statistical Learning  
## Notebook 03_11 — Factor Orthogonalization and Eviction

### Research question

How can a quantitative research process manage a library of alpha factors by removing redundant exposures, measuring incremental value, and evicting stale or duplicated signals?

This notebook follows naturally from:

```text
03_09_volatility_surface_alpha_signals
03_10_statistical_arbitrage_pairs
```

Earlier notebooks built candidate signals. This notebook asks a more institutional question:

> Once we have many signals, how do we stop the factor library from becoming a noisy, redundant factor zoo?

It covers:

1. synthetic cross-sectional equity/futures universe;
2. candidate factor generation;
3. forward-return targets;
4. information coefficient analysis;
5. factor correlation and redundancy;
6. cross-sectional neutralisation;
7. factor orthogonalization by residualisation;
8. incremental IC and incremental regression tests;
9. variance inflation factors;
10. turnover and implementation-cost proxies;
11. rolling IC decay;
12. factor admission rules;
13. factor eviction rules;
14. composite alpha construction;
15. before/after library diagnostics;
16. limitations and research extensions.

The key idea:

> A good factor library is not the one with the most signals. It is the one where each retained factor adds robust, distinct, and implementable information.

## 1. Factor library problem

Suppose we have candidate factors:

$$
f_{1,t}, f_{2,t}, \dots, f_{K,t}
$$

Each factor is a cross-sectional signal across assets at time $t$.

Examples:

- momentum;
- value;
- carry;
- volatility;
- quality;
- skew;
- liquidity;
- microstructure imbalance;
- term-structure features.

A common research failure is to keep adding factors without asking:

1. Is the new factor just a repackaged old factor?
2. Does it add incremental predictive power?
3. Is it stable out of sample?
4. Does it survive costs?
5. Has an existing factor decayed?
6. Should the old factor be evicted?

This notebook builds a systematic answer.

## 2. Cross-sectional information coefficient

For each date $t$, the information coefficient is:

$$
IC_t = corr(f_{i,t}, r_{i,t+1})
$$

where:

- $f_{i,t}$ is the factor value for asset $i$;
- $r_{i,t+1}$ is the forward return.

A positive average IC means higher factor scores tend to predict higher future returns.

Rank IC uses Spearman correlation instead of Pearson correlation.

IC is not a backtest, but it is a clean first diagnostic.

## 3. Neutralisation

A factor may accidentally load on unwanted exposures:

- sector;
- market beta;
- size;
- volatility;
- liquidity.

A neutralised factor removes those exposures by cross-sectional regression:

$$
f_t = X_t\beta_t + \epsilon_t
$$

The residual:

$$
f_t^{neutral} = \epsilon_t
$$

is the factor after removing the controls.

This notebook neutralises against:

1. sector dummies;
2. log size;
3. beta;
4. volatility.

## 4. Orthogonalization

Suppose a new candidate factor $g$ is highly correlated with existing factors $F$.

We can orthogonalize by regression:

$$
g_t = F_t\beta_t + \epsilon_t
$$

The orthogonalized factor is:

$$
g_t^\perp = \epsilon_t
$$

If $g_t^\perp$ has little predictive power, then the candidate factor is probably redundant.

Orthogonalization does not create alpha. It tests whether the factor has **incremental** information beyond the existing library.

## 5. Factor eviction

A factor may be evicted if it is:

1. redundant with another factor;
2. no longer predictive;
3. unstable across regimes;
4. too costly to trade;
5. highly collinear with the composite;
6. data-mined;
7. implementation-heavy with little incremental value.

A possible eviction score:

$$
\begin{aligned}
eviction\_score &= w_1 redundancy \\
&\quad + w_2 decay \\
&\quad + w_3 cost \\
&\quad - w_4 incremental\_IC
\end{aligned}
$$

The purpose is governance:

> A factor library should have a maintenance process, not only a discovery process.

## 6. Imports and configuration

In [ ]:
from __future__ import annotations

from dataclasses import asdict, dataclass
from pathlib import Path
import json

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

In [ ]:
@dataclass(frozen=True)
class FactorLibraryConfig:
    n_dates: int = 900
    n_assets: int = 120
    n_sectors: int = 8
    train_fraction: float = 0.60
    validation_fraction: float = 0.20
    annualisation: int = 252
    ic_window: int = 126
    turnover_cost_per_unit: float = 0.0005
    redundancy_corr_threshold: float = 0.85
    min_test_ic_to_keep: float = 0.01
    max_turnover_to_keep: float = 0.60
    seed: int = 42


config = FactorLibraryConfig()
config

## 7. Synthetic cross-sectional universe

We simulate:

- dates;
- assets;
- sectors;
- size;
- beta;
- volatility;
- latent true alphas;
- future returns.

Candidate factors include:

1. momentum;
2. value;
3. quality;
4. low volatility;
5. carry;
6. liquidity;
7. redundant momentum clone;
8. decayed factor;
9. noisy factor;
10. candidate unique factor.

This gives a controlled setup for testing factor retention and eviction.

In [ ]:
def simulate_factor_universe(config: FactorLibraryConfig) -> pd.DataFrame:
    rng = np.random.default_rng(config.seed)

    dates = pd.bdate_range("2018-01-01", periods=config.n_dates)
    assets = [f"asset_{i:03d}" for i in range(config.n_assets)]
    sectors = rng.integers(0, config.n_sectors, size=config.n_assets)

    base_size = rng.normal(0.0, 1.0, size=config.n_assets)
    base_beta = rng.normal(1.0, 0.25, size=config.n_assets)
    base_vol = rng.lognormal(mean=-4.5, sigma=0.35, size=config.n_assets)

    rows = []

    # Persistent latent styles by asset.
    latent_value = rng.normal(0, 1, config.n_assets)
    latent_quality = rng.normal(0, 1, config.n_assets)
    latent_carry = rng.normal(0, 1, config.n_assets)
    latent_unique = rng.normal(0, 1, config.n_assets)

    for t, date in enumerate(dates):
        market_state = rng.normal(0, 1)
        sector_shock = rng.normal(0, 0.005, config.n_sectors)

        # Time-varying factor premia.
        premium_momentum = 0.0018 + 0.0006 * np.sin(2 * np.pi * t / 252)
        premium_value = 0.0012
        premium_quality = 0.0009
        premium_lowvol = 0.0010
        premium_carry = 0.0011
        premium_unique = 0.0014

        # Decayed factor is useful early, then fades.
        premium_decayed = 0.0018 if t < int(0.55 * config.n_dates) else -0.0001

        for i, asset in enumerate(assets):
            sector = sectors[i]

            size = base_size[i] + 0.05 * rng.normal()
            beta = base_beta[i] + 0.03 * market_state + 0.03 * rng.normal()
            vol = base_vol[i] * np.exp(0.15 * market_state + 0.10 * rng.normal())

            momentum = (
                0.70 * latent_unique[i]
                + 0.30 * rng.normal()
                + 0.20 * np.sin(2 * np.pi * t / 63)
            )
            value = latent_value[i] + 0.35 * rng.normal()
            quality = latent_quality[i] + 0.35 * rng.normal()
            low_vol = -np.log(vol) + 0.20 * rng.normal()
            carry = latent_carry[i] + 0.35 * rng.normal()
            liquidity = -size + 0.50 * rng.normal()

            # Redundant clone of momentum.
            momentum_clone = 0.92 * momentum + 0.20 * rng.normal()

            # Decayed signal.
            decayed_factor = latent_value[i] + 0.50 * rng.normal()

            # Pure noise factor.
            noise_factor = rng.normal()

            # Candidate unique factor with genuine incremental signal.
            unique_factor = latent_unique[i] + 0.45 * rng.normal()

            # Forward return model.
            forward_return = (
                0.0001
                + premium_momentum * momentum
                + premium_value * value
                + premium_quality * quality
                + premium_lowvol * low_vol
                + premium_carry * carry
                + premium_decayed * decayed_factor
                + premium_unique * unique_factor
                + 0.0008 * beta * market_state
                + sector_shock[sector]
                + vol * rng.standard_t(df=6) * np.sqrt((6 - 2) / 6)
            )

            # Realised current return is noisy and loosely related to momentum.
            current_return = 0.25 * forward_return + vol * rng.standard_normal()

            rows.append({
                "date": date,
                "asset": asset,
                "sector": int(sector),
                "log_size": size,
                "beta": beta,
                "volatility": vol,
                "return_t": current_return,
                "forward_return_1d": forward_return,
                "momentum": momentum,
                "value": value,
                "quality": quality,
                "low_vol": low_vol,
                "carry": carry,
                "liquidity": liquidity,
                "momentum_clone": momentum_clone,
                "decayed_factor": decayed_factor,
                "noise_factor": noise_factor,
                "unique_candidate": unique_factor
            })

    return pd.DataFrame(rows)


panel = simulate_factor_universe(config)

panel.head()

In [ ]:
factor_cols = [
    "momentum",
    "value",
    "quality",
    "low_vol",
    "carry",
    "liquidity",
    "momentum_clone",
    "decayed_factor",
    "noise_factor",
    "unique_candidate"
]

control_cols = ["log_size", "beta", "volatility"]

pd.Series({
    "n_rows": len(panel),
    "n_dates": panel["date"].nunique(),
    "n_assets": panel["asset"].nunique(),
    "n_factors": len(factor_cols)
})

## 8. Cross-sectional winsorisation and standardisation

For each date, we:

1. winsorise factor values by cross-sectional quantiles;
2. z-score within date.

This makes factors comparable and reduces outlier influence.

$$
z_{i,t}=\frac{x_{i,t}-\bar x_t}{s_t}
$$

In [ ]:
def cross_sectional_winsorize_zscore(df, cols, lower=0.01, upper=0.99):
    out = df.copy()

    for col in cols:
        z_col = f"z_{col}"

        values = []
        for date, g in out.groupby("date"):
            x = g[col].copy()
            lo = x.quantile(lower)
            hi = x.quantile(upper)
            x = x.clip(lo, hi)
            std = x.std(ddof=1)
            z = (x - x.mean()) / (std if std > 0 else 1.0)
            values.append(pd.Series(z.values, index=g.index))

        out[z_col] = pd.concat(values).sort_index()

    return out


panel_z = cross_sectional_winsorize_zscore(panel, factor_cols + control_cols)

z_factor_cols = [f"z_{c}" for c in factor_cols]
z_control_cols = [f"z_{c}" for c in control_cols]

panel_z[["date", "asset"] + z_factor_cols[:5]].head()

## 9. Chronological split

We split by dates:

1. train;
2. validation;
3. test.

Factor decisions should be made using train/validation, then evaluated on test.

In [ ]:
unique_dates = np.array(sorted(panel_z["date"].unique()))
n_dates = len(unique_dates)

train_end = int(config.train_fraction * n_dates)
validation_end = int((config.train_fraction + config.validation_fraction) * n_dates)

train_dates = unique_dates[:train_end]
validation_dates = unique_dates[train_end:validation_end]
test_dates = unique_dates[validation_end:]

train = panel_z[panel_z["date"].isin(train_dates)].copy()
validation = panel_z[panel_z["date"].isin(validation_dates)].copy()
test = panel_z[panel_z["date"].isin(test_dates)].copy()

split_metadata = {
    "n_dates_total": int(n_dates),
    "n_dates_train": int(len(train_dates)),
    "n_dates_validation": int(len(validation_dates)),
    "n_dates_test": int(len(test_dates)),
    "train_start": str(pd.Timestamp(train_dates[0])),
    "train_end": str(pd.Timestamp(train_dates[-1])),
    "validation_start": str(pd.Timestamp(validation_dates[0])),
    "validation_end": str(pd.Timestamp(validation_dates[-1])),
    "test_start": str(pd.Timestamp(test_dates[0])),
    "test_end": str(pd.Timestamp(test_dates[-1])),
}

pd.Series(split_metadata)

## 10. Information coefficient by factor

For each date and factor:

$$
IC_t = corr(f_{i,t}, r_{i,t+1})
$$

We compute:

- mean IC;
- IC volatility;
- IC information ratio;
- hit rate;
- t-style statistic.

This is the first filter for factor usefulness.

In [ ]:
def daily_ic_table(df, factors, target="forward_return_1d", method="pearson"):
    rows = []

    for date, g in df.groupby("date"):
        for factor in factors:
            if method == "spearman":
                ic = g[factor].rank().corr(g[target].rank())
            else:
                ic = g[factor].corr(g[target])

            rows.append({
                "date": date,
                "factor": factor,
                "ic": ic
            })

    return pd.DataFrame(rows)


def summarize_ic(ic_df):
    rows = []

    for factor, g in ic_df.groupby("factor"):
        ic = g["ic"].dropna()
        mean_ic = ic.mean()
        std_ic = ic.std(ddof=1)
        n = len(ic)

        rows.append({
            "factor": factor,
            "n_dates": n,
            "mean_ic": mean_ic,
            "std_ic": std_ic,
            "ic_ir": mean_ic / std_ic * np.sqrt(config.annualisation) if std_ic > 0 else np.nan,
            "hit_rate_positive_ic": (ic > 0).mean(),
            "t_style_stat": mean_ic / (std_ic / np.sqrt(n)) if std_ic > 0 and n > 1 else np.nan
        })

    return pd.DataFrame(rows).sort_values("mean_ic", ascending=False)


ic_train_daily = daily_ic_table(train, z_factor_cols)
ic_validation_daily = daily_ic_table(validation, z_factor_cols)
ic_test_daily = daily_ic_table(test, z_factor_cols)

ic_train_summary = summarize_ic(ic_train_daily)
ic_validation_summary = summarize_ic(ic_validation_daily)
ic_test_summary = summarize_ic(ic_test_daily)

ic_train_summary

In [ ]:
ic_test_summary

In [ ]:
plt.figure(figsize=(12, 6))
plot_df = ic_test_summary.sort_values("mean_ic")
plt.barh(plot_df["factor"], plot_df["mean_ic"])
plt.axvline(0, linestyle="--")
plt.title("Test Mean IC by Factor")
plt.xlabel("Mean daily IC")
plt.ylabel("Factor")
plt.show()

## 11. Factor correlation and redundancy

A factor can have positive IC but still be redundant.

We compute average cross-sectional factor correlation across dates:

$$
corr_t(f_a, f_b)
$$

then average through time.

High absolute correlation suggests duplication.

In [ ]:
def average_factor_corr(df, factors):
    corr_mats = []

    for date, g in df.groupby("date"):
        corr_mats.append(g[factors].corr().values)

    avg_corr = np.nanmean(np.stack(corr_mats), axis=0)

    return pd.DataFrame(avg_corr, index=factors, columns=factors)


avg_corr_train = average_factor_corr(train, z_factor_cols)

avg_corr_train

In [ ]:
plt.figure(figsize=(9, 8))
plt.imshow(avg_corr_train.values, vmin=-1, vmax=1)
plt.colorbar(label="Average cross-sectional correlation")
plt.xticks(range(len(z_factor_cols)), z_factor_cols, rotation=90)
plt.yticks(range(len(z_factor_cols)), z_factor_cols)
plt.title("Average Factor Correlation, Train")
plt.tight_layout()
plt.show()

In [ ]:
redundant_pairs = []

for i, f1 in enumerate(z_factor_cols):
    for j, f2 in enumerate(z_factor_cols):
        if j <= i:
            continue
        corr = avg_corr_train.loc[f1, f2]
        if abs(corr) >= config.redundancy_corr_threshold:
            redundant_pairs.append({
                "factor_1": f1,
                "factor_2": f2,
                "avg_corr_train": corr
            })

redundant_pairs = pd.DataFrame(redundant_pairs).sort_values("avg_corr_train", ascending=False)

redundant_pairs

## 12. Neutralisation against controls

We neutralise each factor cross-sectionally against:

1. sector dummies;
2. log size;
3. beta;
4. volatility.

For each date:

$$
f_t = X_t\gamma_t+\epsilon_t
$$

Neutralised factor:

$$
f_t^{neutral}=\epsilon_t
$$

This removes broad exposures that may dominate a factor's apparent IC.

In [ ]:
def build_control_matrix(g, z_control_cols, sector_col="sector"):
    sector_dummies = pd.get_dummies(g[sector_col].astype(int), prefix="sector", drop_first=True)
    controls = pd.concat([
        g[z_control_cols].reset_index(drop=True),
        sector_dummies.reset_index(drop=True)
    ], axis=1)

    X = np.column_stack([np.ones(len(g)), controls.to_numpy(dtype=float)])
    return X


def neutralize_factors(df, factors, z_control_cols):
    out = df.copy()

    for factor in factors:
        out[f"{factor}_neutral"] = np.nan

    for date, g in out.groupby("date"):
        idx = g.index
        X = build_control_matrix(g, z_control_cols)

        for factor in factors:
            y = g[factor].to_numpy(dtype=float)
            beta = np.linalg.pinv(X.T @ X) @ X.T @ y
            residual = y - X @ beta

            std = residual.std(ddof=1)
            residual_z = (residual - residual.mean()) / (std if std > 0 else 1.0)

            out.loc[idx, f"{factor}_neutral"] = residual_z

    return out


panel_neutral = neutralize_factors(panel_z, z_factor_cols, z_control_cols)

neutral_factor_cols = [f"{f}_neutral" for f in z_factor_cols]

train_n = panel_neutral[panel_neutral["date"].isin(train_dates)].copy()
validation_n = panel_neutral[panel_neutral["date"].isin(validation_dates)].copy()
test_n = panel_neutral[panel_neutral["date"].isin(test_dates)].copy()

train_n[["date", "asset"] + neutral_factor_cols[:5]].head()

## 13. IC after neutralisation

A factor with high raw IC but low neutralised IC may mostly be exposure to size, beta, volatility, or sector.

That may still be useful, but it should be labelled correctly.

In [ ]:
ic_neutral_train = summarize_ic(daily_ic_table(train_n, neutral_factor_cols))
ic_neutral_validation = summarize_ic(daily_ic_table(validation_n, neutral_factor_cols))
ic_neutral_test = summarize_ic(daily_ic_table(test_n, neutral_factor_cols))

neutral_ic_comparison = (
    ic_test_summary[["factor", "mean_ic", "ic_ir"]]
    .rename(columns={"mean_ic": "raw_test_mean_ic", "ic_ir": "raw_test_ic_ir"})
    .merge(
        ic_neutral_test[["factor", "mean_ic", "ic_ir"]].rename(columns={
            "factor": "neutral_factor",
            "mean_ic": "neutral_test_mean_ic",
            "ic_ir": "neutral_test_ic_ir"
        }),
        left_on="factor",
        right_on="neutral_factor",
        how="left"
    )
)

neutral_ic_comparison

## 14. Orthogonalization to existing factor library

Suppose the current approved factor library is:

```text
momentum, value, quality, low_vol, carry
```

Candidate factors:

```text
liquidity, momentum_clone, decayed_factor, noise_factor, unique_candidate
```

For each candidate, we regress it on the approved library:

$$
g_t = F_t\beta_t+\epsilon_t
$$

The residual:

$$
g_t^\perp=\epsilon_t
$$

is the candidate's incremental component.

If the residual has no IC, the candidate is redundant or useless.

In [ ]:
approved_base = [
    "z_momentum_neutral",
    "z_value_neutral",
    "z_quality_neutral",
    "z_low_vol_neutral",
    "z_carry_neutral"
]

candidate_factors = [
    "z_liquidity_neutral",
    "z_momentum_clone_neutral",
    "z_decayed_factor_neutral",
    "z_noise_factor_neutral",
    "z_unique_candidate_neutral"
]


def orthogonalize_candidates(df, approved_factors, candidates):
    out = df.copy()

    for cand in candidates:
        out[f"{cand}_orth"] = np.nan

    for date, g in out.groupby("date"):
        idx = g.index
        X = np.column_stack([np.ones(len(g)), g[approved_factors].to_numpy(dtype=float)])

        for cand in candidates:
            y = g[cand].to_numpy(dtype=float)
            beta = np.linalg.pinv(X.T @ X) @ X.T @ y
            residual = y - X @ beta

            std = residual.std(ddof=1)
            residual_z = (residual - residual.mean()) / (std if std > 0 else 1.0)

            out.loc[idx, f"{cand}_orth"] = residual_z

    return out


panel_orth = orthogonalize_candidates(panel_neutral, approved_base, candidate_factors)

orth_candidate_cols = [f"{c}_orth" for c in candidate_factors]

train_o = panel_orth[panel_orth["date"].isin(train_dates)].copy()
validation_o = panel_orth[panel_orth["date"].isin(validation_dates)].copy()
test_o = panel_orth[panel_orth["date"].isin(test_dates)].copy()

train_o[["date", "asset"] + orth_candidate_cols].head()

In [ ]:
orth_ic_train = summarize_ic(daily_ic_table(train_o, orth_candidate_cols))
orth_ic_validation = summarize_ic(daily_ic_table(validation_o, orth_candidate_cols))
orth_ic_test = summarize_ic(daily_ic_table(test_o, orth_candidate_cols))

orth_ic_test

## 15. Incremental regression test

IC is univariate.

A stronger test asks:

> Does the candidate improve a cross-sectional regression after existing approved factors are included?

For each date:

$$
\begin{aligned}
r_{i,t+1} &= a_t \\
&\quad + F_{i,t}\beta_t \\
&\quad + \gamma_t g_{i,t} \\
&\quad + \epsilon_{i,t}
\end{aligned}
$$

We track the candidate's coefficient $\gamma_t$ across dates.

If $\gamma_t$ is consistently positive, the factor may add incremental value.

In [ ]:
def daily_incremental_regression(df, approved_factors, candidate, target="forward_return_1d"):
    rows = []

    for date, g in df.groupby("date"):
        X = np.column_stack([
            np.ones(len(g)),
            g[approved_factors].to_numpy(dtype=float),
            g[candidate].to_numpy(dtype=float)
        ])
        y = g[target].to_numpy(dtype=float)

        beta = np.linalg.pinv(X.T @ X) @ X.T @ y
        residual = y - X @ beta

        rows.append({
            "date": date,
            "candidate": candidate,
            "candidate_coef": beta[-1],
            "residual_std": residual.std(ddof=1)
        })

    return pd.DataFrame(rows)


incremental_frames = []

for cand in orth_candidate_cols:
    incremental_frames.append(
        daily_incremental_regression(test_o, approved_base, cand)
    )

incremental_coefs = pd.concat(incremental_frames, ignore_index=True)

incremental_summary = (
    incremental_coefs
    .groupby("candidate")
    .agg(
        n_dates=("candidate_coef", "count"),
        mean_coef=("candidate_coef", "mean"),
        std_coef=("candidate_coef", "std"),
        hit_rate_positive=("candidate_coef", lambda x: (x > 0).mean())
    )
    .reset_index()
)

incremental_summary["t_style_stat"] = (
    incremental_summary["mean_coef"]
    / (incremental_summary["std_coef"] / np.sqrt(incremental_summary["n_dates"]))
)

incremental_summary.sort_values("mean_coef", ascending=False)

## 16. Variance inflation factor

Collinearity can make a factor unstable.

Variance inflation factor is:

$$
VIF_j = \frac{1}{1-R_j^2}
$$

where $R_j^2$ comes from regressing factor $j$ on the other factors.

High VIF suggests redundancy or unstable coefficients.

In [ ]:
def compute_vif(df, factors):
    # Use stacked panel rows as an approximation.
    X = df[factors].dropna().to_numpy(dtype=float)

    rows = []

    for j, factor in enumerate(factors):
        y = X[:, j]
        X_other = np.delete(X, j, axis=1)
        X_design = np.column_stack([np.ones(len(X_other)), X_other])

        beta = np.linalg.pinv(X_design.T @ X_design) @ X_design.T @ y
        fitted = X_design @ beta

        ssr = np.sum((y - fitted) ** 2)
        sst = np.sum((y - y.mean()) ** 2)
        r2 = 1 - ssr / sst if sst > 0 else 0.0
        vif = 1.0 / max(1e-8, 1 - r2)

        rows.append({
            "factor": factor,
            "r2_against_others": r2,
            "vif": vif
        })

    return pd.DataFrame(rows).sort_values("vif", ascending=False)


full_neutral_library = approved_base + candidate_factors
vif_report = compute_vif(train_n, full_neutral_library)

vif_report

## 17. Turnover proxy

A factor may be predictive but too expensive to trade.

We approximate factor turnover by rank changes:

$$
turnover_t = \frac{1}{N} \sum_i |w_{i,t}-w_{i,t-1}|
$$

where weights are proportional to demeaned factor ranks.

This is not a full trading backtest, but it flags fast-moving or noisy factors.

In [ ]:
def factor_rank_weights(g, factor):
    ranks = g[factor].rank(pct=True).to_numpy()
    raw = ranks - np.mean(ranks)
    denom = np.sum(np.abs(raw))
    if denom == 0:
        return raw
    return raw / denom


def factor_turnover(df, factors):
    rows = []

    for factor in factors:
        prev_weights = None

        for date, g in df.groupby("date"):
            g = g.sort_values("asset")
            weights = factor_rank_weights(g, factor)

            if prev_weights is not None:
                turnover = np.sum(np.abs(weights - prev_weights))
                rows.append({
                    "date": date,
                    "factor": factor,
                    "turnover": turnover
                })

            prev_weights = weights

    return pd.DataFrame(rows)


turnover_daily = factor_turnover(test_n, full_neutral_library)
turnover_summary = (
    turnover_daily
    .groupby("factor")
    .agg(
        mean_turnover=("turnover", "mean"),
        median_turnover=("turnover", "median"),
        p90_turnover=("turnover", lambda x: x.quantile(0.90))
    )
    .reset_index()
    .sort_values("mean_turnover", ascending=False)
)

turnover_summary

## 18. Rolling IC decay

A factor can decay through time.

We compute rolling mean IC:

$$
\overline{IC}_{t,w} = \frac{1}{w}\sum_{j=t-w+1}^{t}IC_j
$$

A factor with strong early IC but weak recent IC may be a candidate for eviction.

In [ ]:
def rolling_ic_summary(ic_daily, window):
    rows = []

    for factor, g in ic_daily.groupby("factor"):
        g = g.sort_values("date").copy()
        g["rolling_mean_ic"] = g["ic"].rolling(window, min_periods=max(20, window // 4)).mean()
        rows.append(g)

    return pd.concat(rows, ignore_index=True)


all_ic_daily = daily_ic_table(panel_neutral, full_neutral_library)
rolling_ic = rolling_ic_summary(all_ic_daily, config.ic_window)

rolling_ic.tail()

In [ ]:
plt.figure(figsize=(12, 6))
for factor in ["z_momentum_neutral", "z_momentum_clone_neutral", "z_decayed_factor_neutral", "z_unique_candidate_neutral", "z_noise_factor_neutral"]:
    g = rolling_ic[rolling_ic["factor"] == factor]
    plt.plot(g["date"], g["rolling_mean_ic"], label=factor)
plt.axhline(0, linestyle="--")
plt.title("Rolling Mean IC")
plt.xlabel("Date")
plt.ylabel("Rolling IC")
plt.legend()
plt.show()

## 19. Factor eviction score

We combine diagnostics into an eviction table.

A factor is more likely to be evicted if:

- test IC is weak;
- recent rolling IC is weak;
- turnover is high;
- redundancy is high;
- VIF is high.

This is not a universal formula. It is a governance template.

In [ ]:
def latest_rolling_ic(rolling_ic_df):
    rows = []
    for factor, g in rolling_ic_df.groupby("factor"):
        g = g.sort_values("date")
        rows.append({
            "factor": factor,
            "latest_rolling_ic": g["rolling_mean_ic"].dropna().iloc[-1] if g["rolling_mean_ic"].notna().any() else np.nan
        })
    return pd.DataFrame(rows)


def max_abs_corr_to_others(corr_matrix):
    rows = []
    for factor in corr_matrix.index:
        vals = corr_matrix.loc[factor].drop(factor)
        rows.append({
            "factor": factor,
            "max_abs_corr_to_other": vals.abs().max()
        })
    return pd.DataFrame(rows)


neutral_corr_train = average_factor_corr(train_n, full_neutral_library)
redundancy_report = max_abs_corr_to_others(neutral_corr_train)

ic_test_full = summarize_ic(daily_ic_table(test_n, full_neutral_library))
latest_ic = latest_rolling_ic(rolling_ic)

eviction_table = (
    ic_test_full[["factor", "mean_ic", "ic_ir", "hit_rate_positive_ic"]]
    .merge(latest_ic, on="factor", how="left")
    .merge(turnover_summary[["factor", "mean_turnover"]], on="factor", how="left")
    .merge(redundancy_report, on="factor", how="left")
    .merge(vif_report[["factor", "vif"]], on="factor", how="left")
)

# Normalised components.
eviction_table["weak_ic_score"] = (-eviction_table["mean_ic"]).rank(pct=True)
eviction_table["decay_score"] = (-eviction_table["latest_rolling_ic"]).rank(pct=True)
eviction_table["turnover_score"] = eviction_table["mean_turnover"].rank(pct=True)
eviction_table["redundancy_score"] = eviction_table["max_abs_corr_to_other"].rank(pct=True)
eviction_table["vif_score"] = eviction_table["vif"].rank(pct=True)

eviction_table["eviction_score"] = (
    0.30 * eviction_table["weak_ic_score"]
    + 0.25 * eviction_table["decay_score"]
    + 0.20 * eviction_table["redundancy_score"]
    + 0.15 * eviction_table["turnover_score"]
    + 0.10 * eviction_table["vif_score"]
)

eviction_table["evict_flag"] = (
    (eviction_table["mean_ic"] < config.min_test_ic_to_keep)
    | (eviction_table["max_abs_corr_to_other"] > config.redundancy_corr_threshold)
    | (eviction_table["mean_turnover"] > config.max_turnover_to_keep)
)

eviction_table = eviction_table.sort_values("eviction_score", ascending=False)

eviction_table

In [ ]:
plt.figure(figsize=(12, 6))
plot_df = eviction_table.sort_values("eviction_score")
plt.barh(plot_df["factor"], plot_df["eviction_score"])
plt.title("Factor Eviction Score")
plt.xlabel("Eviction score")
plt.ylabel("Factor")
plt.show()

## 20. Admission test for a new candidate

Candidate:

```text
unique_candidate
```

We ask:

1. Is raw IC positive?
2. Is neutralised IC positive?
3. Is orthogonalized IC positive?
4. Does it improve incremental regression?
5. Is turnover reasonable?
6. Is redundancy low?

If yes, admit.

If no, reject or keep under observation.

In [ ]:
candidate_name = "z_unique_candidate_neutral"
candidate_orth = "z_unique_candidate_neutral_orth"

candidate_admission = {
    "candidate": candidate_name,
    "raw_test_ic": float(ic_test_summary.loc[ic_test_summary["factor"] == "z_unique_candidate", "mean_ic"].iloc[0]),
    "neutral_test_ic": float(ic_neutral_test.loc[ic_neutral_test["factor"] == candidate_name, "mean_ic"].iloc[0]),
    "orthogonal_test_ic": float(orth_ic_test.loc[orth_ic_test["factor"] == candidate_orth, "mean_ic"].iloc[0]),
    "incremental_mean_coef": float(incremental_summary.loc[incremental_summary["candidate"] == candidate_orth, "mean_coef"].iloc[0]),
    "turnover": float(turnover_summary.loc[turnover_summary["factor"] == candidate_name, "mean_turnover"].iloc[0]),
    "max_abs_corr_to_other": float(redundancy_report.loc[redundancy_report["factor"] == candidate_name, "max_abs_corr_to_other"].iloc[0]),
}

candidate_admission["admit_flag"] = (
    (candidate_admission["neutral_test_ic"] > 0.01)
    and (candidate_admission["orthogonal_test_ic"] > 0.005)
    and (candidate_admission["incremental_mean_coef"] > 0)
    and (candidate_admission["max_abs_corr_to_other"] < config.redundancy_corr_threshold)
)

pd.Series(candidate_admission)

## 21. Composite factor before and after eviction

We create two composites:

### Full composite

Average of all neutralised factors:

$$
F^{full}_t = \frac{1}{K}\sum_k f_{k,t}^{neutral}
$$

### Clean composite

Average of retained factors after eviction.

We compare their ICs and turnover.

In [ ]:
retained_factors = eviction_table.loc[~eviction_table["evict_flag"], "factor"].tolist()
evicted_factors = eviction_table.loc[eviction_table["evict_flag"], "factor"].tolist()

if len(retained_factors) == 0:
    retained_factors = approved_base.copy()

retained_factors, evicted_factors

In [ ]:
def add_composite_factor(df, full_factors, retained_factors):
    out = df.copy()
    out["composite_full"] = out[full_factors].mean(axis=1)
    out["composite_clean"] = out[retained_factors].mean(axis=1)

    # Cross-sectional z-score composites.
    out = cross_sectional_winsorize_zscore(out, ["composite_full", "composite_clean"])
    return out


panel_composite = add_composite_factor(panel_neutral, full_neutral_library, retained_factors)

test_c = panel_composite[panel_composite["date"].isin(test_dates)].copy()

composite_ic = summarize_ic(daily_ic_table(test_c, ["z_composite_full", "z_composite_clean"]))

composite_ic

In [ ]:
composite_turnover = factor_turnover(test_c, ["z_composite_full", "z_composite_clean"])
composite_turnover_summary = (
    composite_turnover
    .groupby("factor")
    .agg(
        mean_turnover=("turnover", "mean"),
        median_turnover=("turnover", "median")
    )
    .reset_index()
)

composite_turnover_summary

## 22. Toy long-short factor portfolio

For each date and factor:

1. rank assets by factor score;
2. long top quintile;
3. short bottom quintile;
4. equal weight;
5. apply turnover cost proxy.

This is a toy implementation diagnostic, not a production backtest.

In [ ]:
def long_short_factor_backtest(df, factor, target="forward_return_1d", q=0.20, cost_per_turnover=0.0005):
    rows = []
    prev_weights = None

    for date, g in df.groupby("date"):
        g = g.sort_values("asset").copy()
        scores = g[factor]
        n = len(g)
        n_leg = max(1, int(q * n))

        ranks = scores.rank(method="first")

        weights = np.zeros(n)
        long_idx = ranks.nlargest(n_leg).index
        short_idx = ranks.nsmallest(n_leg).index

        idx_to_pos = {idx: pos for pos, idx in enumerate(g.index)}

        for idx in long_idx:
            weights[idx_to_pos[idx]] = 1.0 / n_leg

        for idx in short_idx:
            weights[idx_to_pos[idx]] = -1.0 / n_leg

        forward_returns = g[target].to_numpy()
        gross_return = float(weights @ forward_returns)

        if prev_weights is None:
            turnover = float(np.sum(np.abs(weights)))
        else:
            turnover = float(np.sum(np.abs(weights - prev_weights)))

        cost = cost_per_turnover * turnover
        net_return = gross_return - cost

        rows.append({
            "date": date,
            "factor": factor,
            "gross_return": gross_return,
            "turnover": turnover,
            "cost": cost,
            "net_return": net_return
        })

        prev_weights = weights

    out = pd.DataFrame(rows)
    out["cum_net_return"] = (1 + out["net_return"]).cumprod()
    return out


factor_bt_frames = []

for factor in ["z_composite_full", "z_composite_clean"] + retained_factors[:5]:
    factor_bt_frames.append(
        long_short_factor_backtest(test_c, factor, cost_per_turnover=config.turnover_cost_per_unit)
    )

factor_bt = pd.concat(factor_bt_frames, ignore_index=True)

def factor_bt_summary(bt):
    rows = []
    for factor, g in bt.groupby("factor"):
        r = g["net_return"]
        rows.append({
            "factor": factor,
            "n_dates": len(g),
            "annualised_return": float(r.mean() * config.annualisation),
            "annualised_vol": float(r.std(ddof=1) * np.sqrt(config.annualisation)),
            "information_ratio": float(r.mean() / r.std(ddof=1) * np.sqrt(config.annualisation)) if r.std(ddof=1) > 0 else np.nan,
            "mean_turnover": float(g["turnover"].mean()),
            "total_cost": float(g["cost"].sum()),
            "total_net_return": float((1 + r).prod() - 1)
        })
    return pd.DataFrame(rows).sort_values("information_ratio", ascending=False)

factor_bt_perf = factor_bt_summary(factor_bt)

factor_bt_perf

In [ ]:
plt.figure(figsize=(12, 6))
for factor, g in factor_bt.groupby("factor"):
    if factor in ["z_composite_full", "z_composite_clean"]:
        plt.plot(g["date"], g["cum_net_return"], label=factor)
plt.title("Toy Long-Short Composite Factor Backtest")
plt.xlabel("Date")
plt.ylabel("Cumulative net return")
plt.legend()
plt.show()

## 23. Before/after diagnostics

We compare:

1. full factor library;
2. clean retained library;
3. evicted factors.

A good eviction process should:

- reduce redundancy;
- preserve or improve IC;
- reduce turnover;
- improve composite robustness.

In [ ]:
before_after = pd.DataFrame([
    {
        "library": "full",
        "n_factors": len(full_neutral_library),
        "mean_abs_pairwise_corr": avg_corr_train.loc[full_neutral_library, full_neutral_library].where(
            ~np.eye(len(full_neutral_library), dtype=bool)
        ).abs().stack().mean() if set(full_neutral_library).issubset(avg_corr_train.index) else np.nan,
        "composite_test_ic": float(composite_ic.loc[composite_ic["factor"] == "z_composite_full", "mean_ic"].iloc[0]),
        "composite_turnover": float(composite_turnover_summary.loc[composite_turnover_summary["factor"] == "z_composite_full", "mean_turnover"].iloc[0])
    },
    {
        "library": "clean_retained",
        "n_factors": len(retained_factors),
        "mean_abs_pairwise_corr": neutral_corr_train.loc[retained_factors, retained_factors].where(
            ~np.eye(len(retained_factors), dtype=bool)
        ).abs().stack().mean() if len(retained_factors) > 1 else 0.0,
        "composite_test_ic": float(composite_ic.loc[composite_ic["factor"] == "z_composite_clean", "mean_ic"].iloc[0]),
        "composite_turnover": float(composite_turnover_summary.loc[composite_turnover_summary["factor"] == "z_composite_clean", "mean_turnover"].iloc[0])
    }
])

before_after

## 24. Practical checklist for factor governance

Before admitting a factor:

1. Does it have positive out-of-sample IC?
2. Does it survive neutralisation?
3. Does it add value after orthogonalisation?
4. Is it not redundant?
5. Is turnover reasonable?
6. Is it robust by sector/regime?
7. Is there a plausible economic mechanism?
8. Does it survive implementation costs?

Before evicting a factor:

1. Has IC decayed?
2. Is the factor now redundant?
3. Has turnover become too high?
4. Does it destabilise the composite?
5. Is poor performance temporary or persistent?
6. Does removal improve the portfolio?
7. Is the factor useful in stress even if average IC is weak?

Factor governance should be systematic, not emotional.

## 25. Saving outputs

The notebook saves:

1. synthetic factor panel;
2. standardised factor panel;
3. neutralised factor panel;
4. IC reports;
5. factor correlation matrix;
6. redundant pair report;
7. orthogonal candidate ICs;
8. incremental regression results;
9. VIF report;
10. turnover report;
11. rolling IC table;
12. eviction table;
13. composite factor diagnostics;
14. toy factor backtest;
15. manifest.

In [ ]:
output_dir = Path("data/processed/factor_orthogonalization_and_eviction_v1")
output_dir.mkdir(parents=True, exist_ok=True)

panel_path = output_dir / "synthetic_factor_panel.csv"
panel_z_path = output_dir / "standardized_factor_panel.csv"
panel_neutral_path = output_dir / "neutralized_factor_panel.csv"
panel_orth_path = output_dir / "orthogonalized_factor_panel.csv"
ic_train_path = output_dir / "ic_train_summary.csv"
ic_validation_path = output_dir / "ic_validation_summary.csv"
ic_test_path = output_dir / "ic_test_summary.csv"
neutral_ic_test_path = output_dir / "neutral_ic_test_summary.csv"
factor_corr_path = output_dir / "factor_correlation_train.csv"
redundant_pairs_path = output_dir / "redundant_pairs.csv"
orth_ic_test_path = output_dir / "orthogonal_candidate_ic_test.csv"
incremental_summary_path = output_dir / "incremental_regression_summary.csv"
vif_path = output_dir / "vif_report.csv"
turnover_summary_path = output_dir / "turnover_summary.csv"
rolling_ic_path = output_dir / "rolling_ic.csv"
eviction_path = output_dir / "eviction_table.csv"
candidate_admission_path = output_dir / "candidate_admission.json"
composite_ic_path = output_dir / "composite_ic.csv"
composite_turnover_path = output_dir / "composite_turnover.csv"
before_after_path = output_dir / "before_after_library_diagnostics.csv"
factor_bt_path = output_dir / "toy_factor_backtest.csv"
factor_bt_perf_path = output_dir / "toy_factor_backtest_performance.csv"
manifest_path = output_dir / "manifest.json"

panel.to_csv(panel_path, index=False)
panel_z.to_csv(panel_z_path, index=False)
panel_neutral.to_csv(panel_neutral_path, index=False)
panel_orth.to_csv(panel_orth_path, index=False)
ic_train_summary.to_csv(ic_train_path, index=False)
ic_validation_summary.to_csv(ic_validation_path, index=False)
ic_test_summary.to_csv(ic_test_path, index=False)
ic_neutral_test.to_csv(neutral_ic_test_path, index=False)
avg_corr_train.to_csv(factor_corr_path)
redundant_pairs.to_csv(redundant_pairs_path, index=False)
orth_ic_test.to_csv(orth_ic_test_path, index=False)
incremental_summary.to_csv(incremental_summary_path, index=False)
vif_report.to_csv(vif_path, index=False)
turnover_summary.to_csv(turnover_summary_path, index=False)
rolling_ic.to_csv(rolling_ic_path, index=False)
eviction_table.to_csv(eviction_path, index=False)
Path(candidate_admission_path).write_text(json.dumps(candidate_admission, indent=2, default=str))
composite_ic.to_csv(composite_ic_path, index=False)
composite_turnover_summary.to_csv(composite_turnover_path, index=False)
before_after.to_csv(before_after_path, index=False)
factor_bt.to_csv(factor_bt_path, index=False)
factor_bt_perf.to_csv(factor_bt_perf_path, index=False)

manifest = {
    "dataset_name": "factor_orthogonalization_and_eviction_outputs",
    "schema_version": "factor_orthogonalization_and_eviction_v1",
    "created_at": pd.Timestamp.now(tz="UTC").isoformat(),
    "config": asdict(config),
    "factor_cols": factor_cols,
    "z_factor_cols": z_factor_cols,
    "control_cols": control_cols,
    "approved_base": approved_base,
    "candidate_factors": candidate_factors,
    "retained_factors": retained_factors,
    "evicted_factors": evicted_factors,
    "candidate_admission": candidate_admission,
    "before_after": before_after.to_dict(orient="records"),
    "best_test_ic_factor": ic_test_summary.iloc[0].to_dict(),
    "notes": [
        "Synthetic cross-sectional panel includes true, redundant, decayed, and noisy factors.",
        "Factors are winsorised and z-scored cross-sectionally by date.",
        "Neutralisation removes sector, size, beta, and volatility controls.",
        "Orthogonalisation tests candidate factors against an existing approved library.",
        "Eviction score combines weak IC, IC decay, redundancy, turnover, and VIF.",
        "Toy long-short backtest is diagnostic only, not a production portfolio simulation."
    ]
}

manifest_path.write_text(json.dumps(manifest, indent=2, default=str))

panel_path, eviction_path, before_after_path, factor_bt_perf_path, manifest_path

## 26. Limitations

### 26.1 Synthetic data

The notebook uses synthetic cross-sectional data.

Real factor research requires survivorship-bias-free universes, point-in-time fundamentals, corporate actions, liquidity filters, and realistic transaction costs.

### 26.2 Simplified neutralisation

Neutralisation uses linear cross-sectional residualisation.

Real risk models may include country, industry, currency, beta, style, liquidity, and nonlinear controls.

### 26.3 Orthogonalization is not magic

Residualisation can remove economically meaningful shared information.

It should be used as a diagnostic, not blindly.

### 26.4 Eviction score is subjective

Weights in the eviction score are governance choices.

Different teams may prefer different thresholds.

### 26.5 IC is not a portfolio backtest

IC measures cross-sectional association.

It does not fully account for turnover, constraints, crowding, borrow, capacity, and costs.

### 26.6 Toy backtest

The long-short backtest is simplified and does not model execution, liquidity, or portfolio optimisation.

### 26.7 Factor decay is hard

A weak recent IC may be noise, not permanent decay.

Eviction decisions should consider statistical uncertainty.

## 27. Research frontier and extensions

Important extensions include:

1. **Risk-model neutralisation**  
   Use a full Barra-style or custom risk model.

2. **Hierarchical factor clustering**  
   Group redundant factors and keep representatives.

3. **Bayesian factor decay models**  
   Estimate probability that a factor's IC has decayed.

4. **Online factor governance**  
   Monitor factor health daily.

5. **False discovery control**  
   Adjust for testing many factors.

6. **Transaction-cost-aware factor selection**  
   Use net IC or expected utility after costs.

7. **Regime-conditioned factor libraries**  
   Keep factors that work in specific regimes.

8. **Nonlinear orthogonalization**  
   Residualise using tree/boosting models instead of linear regression.

9. **Factor crowding diagnostics**  
   Use ownership/flow proxies to detect crowded signals.

10. **Chinese futures factor library**  
   Apply governance to carry, momentum, term structure, liquidity, night-session, and L1 microstructure factors.

## 28. Suggested follow-up notebooks

This notebook naturally leads to:

1. **03_12_information_coefficient_analysis**  
   General IC and signal decay framework.

2. **03_13_tree_models_for_return_prediction**  
   Nonlinear factor combination.

3. **03_14_feature_importance_and_shap_for_alpha**  
   Interpret factor models.

4. **04_03_risk_model_factor_exposures**  
   Portfolio risk factor decomposition.

5. **05_01_vectorized_backtest_engine**  
   Backtest factor portfolios with constraints and costs.

6. **07_01_multi_asset_cta_research_pipeline**  
   Use factor governance in a full futures research system.

## 29. Summary

This notebook implemented factor orthogonalization and eviction.

It showed how to:

1. simulate a cross-sectional factor universe;
2. compute raw factor ICs;
3. detect factor redundancy;
4. neutralise factors against controls;
5. orthogonalize candidates against an approved library;
6. test incremental value;
7. compute VIF and turnover diagnostics;
8. monitor rolling IC decay;
9. build an eviction score;
10. admit or reject a new candidate factor;
11. compare full versus clean factor libraries;
12. test toy long-short factor portfolios;
13. save outputs and manifests.

The key computational takeaway:

> Factor research needs library management: neutralise, orthogonalize, monitor, admit, and evict.

The key financial takeaway:

> A factor is valuable only if it adds distinct, stable, implementable information after existing exposures and costs.

## 30. Further reading

- Grinold and Kahn, *Active Portfolio Management.*
- Barra-style equity risk model literature.
- Fama and French factor literature.
- Harvey, Liu, and Zhu on the factor zoo and multiple testing.
- Literature on factor crowding, alpha decay, cross-sectional IC analysis, and transaction-cost-aware factor investing.